#  Cutisia Elite AI - Master Notebook

Ce notebook permet de **déployer** ou **entraîner** le modèle Cutisia Elite AI pour la détection de maladies cutanées.

---

###  Architecture Dual-Mode
| Mode | Modèle | Précision | Où ? |
|------|--------|-----------|------|
|  **Online (Cloud)** | `best_cutisia_v2L.h5` (608 Mo) | ~70% | Google Colab + pyngrok |
|  **Offline (Local)** | `cutisia_mobile.tflite` (12 Mo) | Basique | Directement sur le téléphone |

###  Pré-requis :
1. **GPU activé** : `Exécution > Modifier le type d'exécution > T4 GPU`.
2. Les modèles Elite entraînés sur Kaggle doivent être sur votre **Google Drive**.
3. Un jeton **Ngrok** (dashboard.ngrok.com).

---

## ⚡ Choix du Mode d'Utilisation
- **Section A** :  Déploiement Rapide (modèles déjà entraînés sur Kaggle)
- **Section B** :  Entraînement Complet (si vous voulez ré-entraîner depuis zéro)

---
#  SECTION A : Déploiement Rapide (Recommandé)
### Utilisez les modèles Elite déjà entraînés sur Kaggle.
###  Il suffit de lancer les 3 cellules ci-dessous.

## A1.  Configuration de l'environnement

In [ ]:
# === ÉTAPE A1 : Montage du Drive et Installation ===
from google.colab import drive
import os

# Montage du Google Drive
drive.mount('/content/drive')

# Installation des dépendances serveur
print(' Installation des dépendances...')
!pip install -q fastapi uvicorn pyngrok nest_asyncio python-multipart opencv-python-headless segmentation-models

# Vérification de la présence des modèles sur le Drive
MODELS_DIR = '/content/drive/MyDrive/Cutisia_Elite_AI_Models'
models_to_check = ['best_cutisia_v2L.h5', 'cutisia_heavy_elite.h5', 'cutisia_mobile.tflite']

print('\n Vérification des modèles sur Google Drive...')
for m in models_to_check:
    path = os.path.join(MODELS_DIR, m)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f'  ✅ {m} ({size_mb:.1f} Mo)')
    else:
        print(f'  ❌ {m} - MANQUANT!')

print('\n🎉 Environnement prêt!')

## A2.  Chargement du Modèle Elite

In [ ]:
# === ÉTAPE A2 : Chargement du modèle lourd ===
import numpy as np
from tensorflow.keras.models import load_model

# On charge le meilleur modèle (best > heavy > local)
MODEL_PRIORITY = [
    '/content/drive/MyDrive/Cutisia_Elite_AI_Models/best_cutisia_v2L.h5',
    '/content/drive/MyDrive/Cutisia_Elite_AI_Models/cutisia_heavy_elite.h5',
]

MODEL_PATH = None
for path in MODEL_PRIORITY:
    if os.path.exists(path):
        MODEL_PATH = path
        break

if MODEL_PATH is None:
    raise FileNotFoundError('❌ Aucun modèle trouvé sur le Drive! Vérifiez le dossier Cutisia_Elite_AI_Models.')

print(f' Chargement du modèle : {os.path.basename(MODEL_PATH)}...')
print(f'   Taille : {os.path.getsize(MODEL_PATH) / (1024*1024):.1f} Mo')
model = load_model(MODEL_PATH)
print('✅ Modèle Elite chargé avec succès!')
print(f'   Architecture : EfficientNetV2-L')
print(f'   Classes : Candidiase, Leprosy, Mélanomes, Monkeypox, Scabies, Tinea')

## A3.  Lancement du Serveur API (pyngrok)

In [ ]:
# === ÉTAPE A3 : Lancement du serveur FastAPI + Ngrok + U-Net ===
import io
import os
import cv2
import json
import time
import uvicorn
import nest_asyncio
import tensorflow as tf
import keras
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

# --- PATCH KERAS 3 POUR SEGMENTATION-MODELS ---
class GenericUtilsStub:
    def get_custom_objects(self):
        try: return keras.saving.get_custom_objects()
        except: return {}
if not hasattr(keras.utils, 'generic_utils'):
    keras.utils.generic_utils = GenericUtilsStub()
os.environ['SM_FRAMEWORK'] = 'tf.keras'

!pip install -q segmentation-models
import segmentation_models as sm

# --- CONFIGURATION ---
LABELS = ['Candidiase', 'Leprosy', 'Mélanomes', 'Monkeypox', 'Scabies', 'Tinea']
IMG_SIZE = (384, 384)
UNET_SIZE = (256, 256)
PORT = 8000
NGROK_TOKEN = 'collez_ici_le_token'
COLLECT_DIR = "/content/drive/MyDrive/new_cutisia_images"

# Création du dossier de collecte sur Drive
if not os.path.exists(COLLECT_DIR):
    try:
        os.makedirs(COLLECT_DIR)
        print(f'📁 Dossier de collecte créé : {COLLECT_DIR}')
    except:
        print('⚠️ Impossible de créer le dossier sur Drive, utilisation locale.')
        COLLECT_DIR = 'new_cutisia_images'
        if not os.path.exists(COLLECT_DIR): os.makedirs(COLLECT_DIR)

# --- CHARGEMENT DU MODÈLE U-NET (Segmentation) ---
print('🧠 Chargement de U-Net (segmentation des lésions)...')
BACKBONE = 'resnet34'
unet_preprocess = sm.get_preprocessing(BACKBONE)
unet_model = sm.Unet(BACKBONE, encoder_weights='imagenet', classes=1, activation='sigmoid')
print('✅ Modèle U-Net chargé.')

# --- PRÉTRAITEMENT AVEC U-NET ---
def equalize_derma(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def segment_and_crop(img_rgb):
    """Utilise U-Net pour isoler la lésion et cropper l'image."""
    h, w = img_rgb.shape[:2]
    img_small = cv2.resize(img_rgb, UNET_SIZE)
    img_input = np.expand_dims(img_small, axis=0)
    img_input = unet_preprocess(img_input)
    pr_mask = unet_model.predict(img_input, verbose=0).squeeze()
    binary_mask = (pr_mask > 0.5).astype(np.uint8) * 255
    full_mask = cv2.resize(binary_mask, (w, h))
    coords = np.where(full_mask > 0)
    if len(coords[0]) == 0:
        return img_rgb
    y0, x0 = coords[0].min(), coords[1].min()
    y1, x1 = coords[0].max(), coords[1].max()
    return img_rgb[y0:y1, x0:x1]

def preprocess_image(image):
    img = np.array(image.convert('RGB'))
    img = segment_and_crop(img)      # 1. U-Net : isoler la lésion
    img = cv2.resize(img, IMG_SIZE)   # 2. Resize unique (384x384)
    img = equalize_derma(img)         # 3. CLAHE
    img = img.astype(np.float32) / 255.0  # 4. Normalisation
    return np.expand_dims(img, axis=0)

# --- API ---
app = FastAPI(title='Cutisia Elite API')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/')
def home():
    return {'status': 'online', 'model': 'EfficientNetV2-L', 'segmentation': 'U-Net/ResNet34', 'classes': LABELS}

@app.post('/predict')
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents))
    processed = preprocess_image(image)
    predictions = model.predict(processed)
    idx = int(np.argmax(predictions[0]))
    return {
        'class': LABELS[idx],
        'confidence': float(predictions[0][idx]),
        'results': {LABELS[i]: float(predictions[0][i]) for i in range(len(LABELS))}
    }

@app.post('/collect')
async def collect(file: UploadFile = File(...), metadata: str = Form(...)):
    """Reçoit une image et un JSON, les enregistre sur le Drive"""
    try:
        timestamp = int(time.time() * 1000)
        base_filename = f'collect_{timestamp}'
        
        # 1. Sauvegarde de l'image
        ext = os.path.splitext(file.filename)[1] or '.jpg'
        img_path = os.path.join(COLLECT_DIR, base_filename + ext)
        with open(img_path, 'wb') as f:
            f.write(await file.read())
            
        # 2. Sauvegarde des métadonnées JSON
        json_path = os.path.join(COLLECT_DIR, base_filename + '.json')
        with open(json_path, 'w', encoding='utf-8') as f:
            f.write(metadata)
            
        return {'status': 'success', 'message': 'Données enregistrées', 'filename': base_filename}
    except Exception as e:
        return {'status': 'error', 'message': str(e)}

# --- LANCEMENT ---
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print('\n' + '='*60)
print('🏥 CUTISIA ELITE AI - SERVEUR EN LIGNE')
print('='*60)
print(f'\n🌍 URL PUBLIQUE : {public_url}')
print(f'🔗 Endpoint API : {public_url}/predict')
print(f'📂 Collecte vers : {COLLECT_DIR}')
print(f'🧠 Pipeline : U-Net (segmentation) → CLAHE → EfficientNetV2 (classification)')
print(f'\n📱 Copiez cette URL dans votre app Flutter :')
print(f'   cloud_api_service.dart > baseUrl = "{public_url}"')
print('\n⚠️  Ne fermez PAS cette cellule tant que vous utilisez l\'API!')
print('='*60 + '\n')

nest_asyncio.apply()
config = uvicorn.Config(app, host='0.0.0.0', port=PORT)
server = uvicorn.Server(config)
await server.serve()

---
#  SECTION B : Entraînement Complet (Optionnel)
### ⚠️ Ne lancez cette section QUE si vous voulez ré-entraîner le modèle depuis zéro.
### Durée estimée : 3h à 6h sur GPU T4.

## B1.  Clonage du Dépôt

In [ ]:
import os
%cd /content/
REPO_URL = 'https://github.com/franck504/skin-disease-detector.git'
if not os.path.exists('skin-disease-detector'):
    !git clone {REPO_URL}

%cd /content/skin-disease-detector
!git pull origin main

## B2. 🔗 Liaison des Données (Drive → Colab)

In [ ]:
DATASET_DRIVE_PATH = '/content/drive/MyDrive/datasets-cutisia'

if not os.path.exists('datasets-cutisia'):
    !ln -s "{DATASET_DRIVE_PATH}" "datasets-cutisia"

!ls datasets-cutisia

## B3.  Prétraitement Médical (Masques & Audit)

In [ ]:
# 📊 Statistiques du dataset
!python3 audit_dataset.py

# 🎭 Génération des Pseudo-Masques (à faire une seule fois)
!python3 generate_masks.py

## B4. 🏋️ Lancement de l'Entraînement

In [ ]:
# ⚠️ ATTENTION : EfficientNetV2-L (Ultra-Précision)
# Entraînement sur 11k images + Fine-tuning progressif.
# Durée estimée : 3h à 6h sur GPU T4.
!python3 train_dual_models.py

## B5. 💾 Sauvegarde des Modèles sur Drive

In [ ]:
import shutil

SAVE_DIR = '/content/drive/MyDrive/Cutisia_Elite_AI_Models'
os.makedirs(SAVE_DIR, exist_ok=True)

# Copie des modèles générés vers le Drive
for f in ['best_cutisia_v2L.h5', 'cutisia_heavy_elite.h5', 'cutisia_mobile.tflite']:
    src = f'/content/skin-disease-detector/{f}'
    if os.path.exists(src):
        shutil.copy2(src, SAVE_DIR)
        print(f'✅ {f} copié vers le Drive')
    else:
        print(f'⚠️ {f} non trouvé (vérifier le script d\'entraînement)')

print(f'\n📁 Modèles sauvegardés dans : {SAVE_DIR}')
!ls -lh "{SAVE_DIR}"/*.h5 "{SAVE_DIR}"/*.tflite

## B6. 🌍 Lancement du Serveur API (Après Entraînement)

In [ ]:
import os

# Configuration Ngrok
os.environ['NGROK_AUTHTOKEN'] = 'collez_ici_le_token'

# Lancement de l'API
!python3 api_server.py